In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# loading CSV
df = pd.read_csv("finances_dataset.csv")

# seperate target features
X = df.drop(columns=["defaulted"])
y = df["defaulted"]

# train val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scale continuous features
numeric_cols = X_train.columns.drop("customer_id")
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

X_train.head(), y_train.head()


(   customer_id       age  tenure_months  monthly_income  monthly_expense  \
 45        C046  0.673123       0.468703        0.642337         0.633208   
 12        C013 -0.385722      -0.369665       -0.188159        -0.295866   
 38        C039 -1.550452      -1.465992       -1.641528        -1.534631   
 8         C009 -1.338683      -1.337012       -1.433904        -1.379786   
 1         C002  0.143700       0.017274        0.278995         0.013826   
 
     credit_score  debt_to_income  late_payments  last_month_transactions  \
 45      0.829221       -0.681591      -0.287055                 0.677451   
 12     -0.054755       -0.149618      -0.287055                -0.340024   
 38     -2.154198        2.244263       2.392128                -2.084266   
 8      -1.684586        1.712290       2.392128                -1.648206   
 1       1.243584       -0.681591      -0.287055                 0.822805   
 
     investment_balance  
 45            0.486004  
 12           -0.392

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf

end = datetime.today()
start = end - timedelta(days=365 * 10)
t = yf.Ticker("TSLA")

# Monthly prices
prices = t.history(start=start.date(), end=end.date(), auto_adjust=False).reset_index()
if prices["Date"].dt.tz is not None:
    prices["Date"] = prices["Date"].dt.tz_localize(None)
prices_m = prices.set_index("Date").resample("ME").last()

# Quarterly statements
fin_q = t.quarterly_financials.T
bal_q = t.quarterly_balance_sheet.T
cash_q = t.quarterly_cashflow.T
for df in (fin_q, bal_q, cash_q):
    df.index = pd.to_datetime(df.index).tz_localize(None)

def pick(df, cols):
    return df[[c for c in cols if c in df.columns]]

fin = pick(fin_q, ["Total Revenue", "Gross Profit", "Operating Income", "Net Income", "Normalized EBITDA"])
bal = pick(bal_q, ["Total Assets", "Total Liab", "Total Debt", "Total Current Assets", "Total Current Liabilities"])
cash = pick(cash_q, ["Total Cash From Operating Activities", "Capital Expenditures"])  # fixed typo

financials = pd.concat([fin, bal, cash], axis=1)

def ratio(num, denom, name):
    if num in financials.columns and denom in financials.columns:
        financials[name] = financials[num] / financials[denom]

ratio("Gross Profit", "Total Revenue", "gross_margin")
ratio("Operating Income", "Total Revenue", "op_margin")
ratio("Net Income", "Total Revenue", "net_margin")
ratio("Total Debt", "Total Assets", "debt_to_assets")
ratio("Total Current Assets", "Total Current Liabilities", "current_ratio")
ratio("Total Cash From Operating Activities", "Total Revenue", "cfo_margin")
ratio("Capital Expenditures", "Total Revenue", "capex_to_revenue")

financials = financials.resample("ME").ffill()
merged = prices_m.join(financials, how="left")
merged = merged.loc[:, ~merged.isna().all()]

print("Columns with data:", merged.columns.tolist())
print(merged.tail(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import classification_report
from sklearn.ensemble import HistGradientBoostingClassifier

